# Clustering Distrital con Grafos — VGAE / DGI

Para cada año (2020–2025) construimos un grafo donde:
- **Nodos** = distritos (ubigeos) con features tabulares normalizadas
- **Aristas** = contigüidad Queen derivada del GeoJSON de polígonos distritales

Luego entrenamos **VGAE** y **DGI** para aprender embeddings no supervisados y aplicamos **K-Means** para encontrar clusters.

## 0. Dependencias

In [ ]:
# !pip install torch-geometric libpysal umap-learn

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.impute import SimpleImputer

import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, VGAE, DeepGraphInfomax
from torch_geometric.utils import from_scipy_sparse_matrix

import libpysal
import umap

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## 1. Cargar datos

In [ ]:
PANEL_PATH   = '../../data/clean/merged/panel_distrital_clean.csv'
GEOJSON_PATH = '../03_causal_model/cghciv.geojson'

panel = pd.read_csv(PANEL_PATH)
gdf   = gpd.read_file(GEOJSON_PATH)
gdf['UBIGEO'] = gdf['UBIGEO'].astype(str).str.zfill(6)

print('Panel:', panel.shape)
print('GeoJSON distritos:', len(gdf))
panel.head(2)

## 2. Definición de features de nodo

In [ ]:
NODE_FEATURES = [
    'prevalencia_anemia',
    'gasto_total', 'gasto_anemia_pan',
    'personal_total', 'programa_anemia', 'centro_salud_municipal',
    'altitude', 'superficie', 'pob_densidad_2020',
    'pct_cultivo', 'pct_construido', 'pct_desnudo', 'pct_agua_visible',
    'n_edificios', 'densidad_edificios_km2',
    'elevacion_media', 'pendiente_media',
    'pct_agua_permanente', 'pct_agua_estacional',
]

print(f'Features de nodo: {len(NODE_FEATURES)}')

## 3. Construcción de aristas — Contigüidad Queen

In [ ]:
def build_edge_index(gdf_sorted):
    """Construye edge_index de PyG a partir de contigüidad Queen."""
    w = libpysal.weights.Queen.from_dataframe(gdf_sorted, silence_warnings=True)
    sp = w.sparse
    edge_index, _ = from_scipy_sparse_matrix(sp)
    return edge_index

# Calcular una vez (la topología no cambia entre años)
gdf_sorted = gdf.sort_values('UBIGEO').reset_index(drop=True)
EDGE_INDEX  = build_edge_index(gdf_sorted)
GEO_UBIGEOS = gdf_sorted['UBIGEO'].tolist()  # orden canónico de nodos

print(f'Nodos en geojson: {len(GEO_UBIGEOS)}')
print(f'Aristas (no dirigidas): {EDGE_INDEX.shape[1] // 2}')

## 4. Función: construir `Data` de PyG por año

In [ ]:
def build_graph(year: int) -> Data:
    df_yr = panel[panel['anio'] == year].copy()
    df_yr['ubigeo_str'] = df_yr['ubigeo'].astype(str).str.zfill(6)
    df_yr = df_yr.set_index('ubigeo_str')

    # Reindexar al orden canónico del geojson
    df_yr = df_yr.reindex(GEO_UBIGEOS)

    X = df_yr[NODE_FEATURES].values.astype(np.float32)

    # Imputar con mediana
    imp = SimpleImputer(strategy='median')
    X = imp.fit_transform(X)

    # Normalizar
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    x = torch.tensor(X, dtype=torch.float)
    data = Data(x=x, edge_index=EDGE_INDEX)
    return data, df_yr

# Prueba
data_test, _ = build_graph(2022)
print('Nodos:', data_test.num_nodes, '| Features:', data_test.num_node_features,
      '| Aristas:', data_test.num_edges)

## 5. Modelos

### 5a. VGAE

In [ ]:
class VGAEEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden=64, out=32):
        super().__init__()
        self.conv_shared = GCNConv(in_channels, hidden)
        self.conv_mu     = GCNConv(hidden, out)
        self.conv_logstd = GCNConv(hidden, out)

    def forward(self, x, edge_index):
        h = F.relu(self.conv_shared(x, edge_index))
        return self.conv_mu(h, edge_index), self.conv_logstd(h, edge_index)


def train_vgae(data: Data, epochs=200, lr=1e-2, hidden=64, out=32):
    data = data.to(DEVICE)
    model = VGAE(VGAEEncoder(data.num_node_features, hidden, out)).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    model.train()
    for epoch in range(1, epochs + 1):
        optimizer.zero_grad()
        z = model.encode(data.x, data.edge_index)
        loss = model.recon_loss(z, data.edge_index) + \
               (1 / data.num_nodes) * model.kl_loss()
        loss.backward()
        optimizer.step()
        if epoch % 50 == 0:
            print(f'  [VGAE] epoch {epoch:3d} | loss {loss.item():.4f}')

    model.eval()
    with torch.no_grad():
        z = model.encode(data.x, data.edge_index)
    return z.cpu().numpy()

### 5b. DGI

In [ ]:
class DGIEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden=512):
        super().__init__()
        self.conv = GCNConv(in_channels, hidden)
        self.prelu = torch.nn.PReLU(hidden)

    def forward(self, x, edge_index):
        return self.prelu(self.conv(x, edge_index))


def corruption(x, edge_index):
    return x[torch.randperm(x.size(0))], edge_index


def train_dgi(data: Data, epochs=300, lr=1e-3, hidden=512):
    data = data.to(DEVICE)
    model = DeepGraphInfomax(
        hidden_channels=hidden,
        encoder=DGIEncoder(data.num_node_features, hidden),
        summary=lambda z, *args, **kwargs: z.mean(dim=0).sigmoid(),
        corruption=corruption,
    ).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    model.train()
    for epoch in range(1, epochs + 1):
        optimizer.zero_grad()
        pos_z, neg_z, summary = model(data.x, data.edge_index)
        loss = model.loss(pos_z, neg_z, summary)
        loss.backward()
        optimizer.step()
        if epoch % 50 == 0:
            print(f'  [DGI]  epoch {epoch:3d} | loss {loss.item():.4f}')

    model.eval()
    with torch.no_grad():
        z, _, _ = model(data.x, data.edge_index)
    return z.cpu().numpy()

## 6. Clustering y evaluación

In [ ]:
def best_kmeans(Z, k_range=range(3, 11)):
    """Selecciona k óptimo por Silhouette Score."""
    scores = {}
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(Z)
        scores[k] = silhouette_score(Z, labels)
    best_k = max(scores, key=scores.get)
    print(f'  Silhouette por k: {scores}')
    print(f'  Mejor k: {best_k} (silhouette={scores[best_k]:.3f})')
    km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
    return km.fit_predict(Z), best_k, scores

## 7. Pipeline completo por año

In [ ]:
YEARS   = [2020, 2021, 2022, 2023, 2024, 2025]
MODEL   = 'VGAE'   # cambiar a 'DGI' para comparar

results = {}  # year -> {labels, Z, best_k, df_yr}

for year in YEARS:
    print(f'\n=== Año {year} | Modelo {MODEL} ===')
    data, df_yr = build_graph(year)

    if MODEL == 'VGAE':
        Z = train_vgae(data)
    else:
        Z = train_dgi(data)

    labels, best_k, sil_scores = best_kmeans(Z)
    results[year] = dict(Z=Z, labels=labels, best_k=best_k, df_yr=df_yr)

print('\nDone.')

## 8. Visualización: UMAP

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, year in enumerate(YEARS):
    Z      = results[year]['Z']
    labels = results[year]['labels']
    k      = results[year]['best_k']

    reducer = umap.UMAP(n_components=2, random_state=42)
    Z2d = reducer.fit_transform(Z)

    scatter = axes[i].scatter(Z2d[:, 0], Z2d[:, 1], c=labels,
                               cmap='tab10', s=8, alpha=0.7)
    axes[i].set_title(f'{year} — {k} clusters ({MODEL})')
    axes[i].axis('off')

plt.suptitle(f'UMAP de embeddings {MODEL} por año', fontsize=14)
plt.tight_layout()
plt.savefig('fig_umap_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Visualización: mapa coroplético por año

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, year in enumerate(YEARS):
    labels = results[year]['labels']
    gdf_plot = gdf_sorted.copy()
    gdf_plot['cluster'] = labels

    gdf_plot.plot(column='cluster', ax=axes[i], cmap='tab10',
                  legend=False, linewidth=0.1, edgecolor='white')
    axes[i].set_title(f'{year}')
    axes[i].axis('off')

plt.suptitle(f'Clusters distritales por año — {MODEL}', fontsize=14)
plt.tight_layout()
plt.savefig('fig_map_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Perfil epidemiológico por cluster

In [ ]:
YEAR_ANALYSIS = 2023

df_yr   = results[YEAR_ANALYSIS]['df_yr'].copy()
labels  = results[YEAR_ANALYSIS]['labels']
df_yr['cluster'] = labels

profile_cols = ['prevalencia_anemia', 'gasto_total', 'gasto_anemia_pan',
                'elevacion_media', 'pob_densidad_2020']

profile = df_yr.groupby('cluster')[profile_cols].mean().round(3)
profile['n_distritos'] = df_yr.groupby('cluster').size()
print(profile.to_string())

## 11. Exportar asignaciones de cluster

In [ ]:
rows = []
for year, res in results.items():
    for ubigeo, cluster in zip(GEO_UBIGEOS, res['labels']):
        rows.append({'ubigeo': ubigeo, 'anio': year, 'cluster': int(cluster)})

df_clusters = pd.DataFrame(rows)
df_clusters.to_csv('../../outputs/clusters_distritales.csv', index=False)
print('Guardado en outputs/clusters_distritales.csv')
df_clusters.head()